In [ ]:
import polars as pl
import numpy as np
import sys
from pathlib import Path

# Configuración de rutas estáticas
sys.path.append("..")
from config.rutas import RUTA_DATA_PROCESSED

# Cargar dataset preprocesado con Polars
df = pl.read_parquet(RUTA_DATA_PROCESSED / "endireh_limpio.parquet")
print("Dimensiones del dataset:", df.shape)
df.head()

In [ ]:
import pandas as pd

clasificacion = [
    {"Variable": "edad_primer_union", "Tipo de Dato": "Cuantitativa Discreta", "Justificación": "Años cumplidos de la mujer al unirse por primera vez."},
    {"Variable": "num_hijos", "Tipo de Dato": "Cuantitativa Discreta", "Justificación": "Conteo entero del número total de hijos nacidos vivos."},
    {"Variable": "nom_entidad", "Tipo de Dato": "Cualitativa Nominal", "Justificación": "Categorías geográficas de las 32 entidades federativas."},
    {"Variable": "nivel_escolaridad", "Tipo de Dato": "Cualitativa Ordinal", "Justificación": "Niveles educativos con jerarquía clara (sin escolaridad, primaria, secundaria, etc.)."},
    {"Variable": "estado_civil_desc", "Tipo de Dato": "Cualitativa Nominal", "Justificación": "Estado conyugal actual de la mujer (casada, soltera, divorciada, etc.)."},
    {"Variable": "sufrio_violencia_pareja", "Tipo de Dato": "Cualitativa Dicotómica", "Justificación": "Indicador binario (1 = Sufrió violencia, 0 = No sufrió violencia)."},
    {"Variable": "factor_expansion", "Tipo de Dato": "Cuantitativa Continua", "Justificación": "Ponderador muestral numérico que representa el número de mujeres que equivale la observación."}
]

df_clasificacion = pd.DataFrame(clasificacion)
df_clasificacion

In [ ]:
# Filtrar registros válidos
df_union_valid = df.with_columns(
    pl.col("edad_primer_union").cast(pl.Float64, strict=False).alias("edad_u_num"),
    pl.col("num_hijos").cast(pl.Float64, strict=False).alias("hijos_num")
).filter(
    (pl.col("edad_u_num") >= 10) & (pl.col("edad_u_num") < 98) &
    (pl.col("hijos_num") >= 0) & (pl.col("hijos_num") < 98)
)

# Medidas simples
resumen_simple = df_union_valid.select([
    pl.col("edad_u_num").mean().alias("media_simple_edad"),
    pl.col("edad_u_num").median().alias("mediana_edad"),
    pl.col("edad_u_num").quantile(0.25).alias("Q1_edad"),
    pl.col("edad_u_num").quantile(0.75).alias("Q3_edad"),
    pl.col("hijos_num").mean().alias("media_simple_hijos"),
    pl.col("hijos_num").median().alias("mediana_hijos")
])

# Medidas ponderadas con NumPy
edades = df_union_valid["edad_u_num"].to_numpy()
hijos = df_union_valid["hijos_num"].to_numpy()
pesos = df_union_valid["factor_expansion"].to_numpy()

media_pond_edad = np.average(edades, weights=pesos)
media_pond_hijos = np.average(hijos, weights=pesos)

print("--- MEDIDAS DE LOCALIZACIÓN ---")
print(resumen_simple)
print(f"\nMedia ponderada de edad a la primera unión: {media_pond_edad:.2f} años")
print(f"Media ponderada de número de hijos: {media_pond_hijos:.2f}")

In [ ]:
def calcular_variabilidad(df_grupo, col_name):
    datos = df_grupo[col_name].to_numpy()
    mean_val = np.mean(datos)
    std_val = np.std(datos, ddof=1)
    q1 = np.percentile(datos, 25)
    q3 = np.percentile(datos, 75)
    
    return {
        "Media": mean_val,
        "Desviación Estándar (s)": std_val,
        "Varianza (s²)": std_val**2,
        "IQR": q3 - q1,
        "CV (%)": (std_val / mean_val) * 100
    }

grupo_con = df_union_valid.filter(pl.col("sufrio_violencia_pareja") == 1)
grupo_sin = df_union_valid.filter(pl.col("sufrio_violencia_pareja") == 0)

metrics_con = calcular_variabilidad(grupo_con, "edad_u_num")
metrics_sin = calcular_variabilidad(grupo_sin, "edad_u_num")

df_comparativo = pd.DataFrame([metrics_con, metrics_sin], index=["Con Violencia", "Sin Violencia"])
print("--- COMPARACIÓN DE DISPERSIÓN DE EDAD A LA PRIMERA UNIÓN ---")
df_comparativo